In [26]:
import pandas as pd
from validation_structure import normaliser, valider, PAYS_COUVERTS
import re

In [27]:
df_tva = pd.read_csv("../data/numeros_tva.csv", dtype=str, keep_default_na=False)

df_tva

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm
...,...,...,...,...,...,...
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp
9998,9999,Prima Distribution AB,LU,LU.98154452,2024-02-15,reprise_erp


**1. Motifs d'invalidité des numéros de TVA :**

In [28]:
df_tva["numero_tva"] = df_tva["numero_tva"].map(normaliser)

In [29]:
df_tva[["tva_valide", "tva_validite"]] = df_tva["numero_tva"].map(valider).tolist()

display(df_tva[df_tva["tva_valide"] == False][["id", "tva_validite"]].groupby("tva_validite").count())
df_tva[df_tva["tva_valide"] == False][["id", "tva_validite"]].groupby("tva_validite").count().sum()

,id
tva_validite,
cle_invalide,2605
pays_non_couvert,1090
trop_court,48
vide,174


id    3917
dtype: int64

In [30]:
df_tva.dtypes

id                   str
raison_sociale       str
pays_declare         str
numero_tva           str
date_saisie          str
source_saisie        str
tva_valide        object
tva_validite         str
dtype: object

In [31]:
df_tva[df_tva["tva_validite"] == "cle_invalide"]

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,False,cle_invalide
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,False,cle_invalide
6,7,Textil Services Oy,NL,NL505862176B88,2024-10-14,saisie_manuelle,False,cle_invalide
13,14,Alpine Logistics Sp. z o.o.,SE,SE1861093621016,2025-05-07,import_fournisseur,False,cle_invalide
14,15,Baltic Industries SAS,PT,PT901258090,2025-09-07,import_fournisseur,False,cle_invalide
...,...,...,...,...,...,...,...,...
9972,9973,Hanse Services Sp. z o.o.,BE,BE078337031,2024-06-20,reprise_erp,False,cle_invalide
9976,9977,Kappa Foods Oy,PT,PT764818934,2025-05-27,import_fournisseur,False,cle_invalide
9977,9978,Batavia Consulting Oy,FI,FI07462360,2024-10-05,crm,False,cle_invalide
9997,9998,Corvus Industries SARL,SE,SE96633532830,2024-09-12,reprise_erp,False,cle_invalide


In [32]:
df_tva_filtre_1 = df_tva[(df_tva["tva_validite"] != "vide") & (df_tva["tva_validite"] != "trop_court") & (df_tva["tva_validite"] != "cle_invalide")]
df_tva_filtre_1

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,True,ok
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,True,ok
3,4,Lumen Distribution Sp. z o.o.,SE,SE751298146001,2024-02-27,import_fournisseur,True,ok
5,6,Kappa Logistics AB,DK,DK61530843,2025-01-14,import_fournisseur,True,ok
7,8,Vertex Industries SARL,SE,SE314125231601,2025-03-31,reprise_erp,True,ok
...,...,...,...,...,...,...,...,...
9993,9994,Fabrica Technologies GmbH,BE,BE1769956921,2025-02-25,import_fournisseur,True,ok
9994,9995,Orion Technologies SAS,PT,PT772064113,2024-03-19,portail_client,True,ok
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur,True,ok
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm,True,ok


**2. Traitement des doublons**

En France le numéro de TVA est composé du SIREN, un SIREN regroupe plusieurs SIRET

In [35]:
df_tva_filtre_1["doublon_tva"] = df_tva_filtre_1["numero_tva"].duplicated()
df_tva_filtre_1[df_tva_filtre_1["doublon_tva"] == True]

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite,doublon_tva
107,108,Vela Materials SpA,NL,NL785427788B39,2024-04-18,crm,True,ok,True
113,114,Granit Industries NV,IT,IT53680520532,2025-03-06,import_fournisseur,True,ok,True
137,138,Lumen Transports BV,PL,PL5102708610,2024-10-03,saisie_manuelle,True,ok,True
155,156,Metallo Transports AB,BE,BE1541337423,2024-07-14,saisie_manuelle,True,ok,True
229,230,Lusitania Transports SARL,ZZ,ZZ23140153,2024-08-01,import_fournisseur,False,pays_non_couvert,True
...,...,...,...,...,...,...,...,...,...
9937,9938,Granit Industries SA,DK,DK48622259,2025-04-01,import_fournisseur,True,ok,True
9989,9990,Kappa Logistics Oy,SE,SE342108454601,2024-04-14,portail_client,True,ok,True
9990,9991,Verso Transports SA,SE,SE604126160001,2025-03-12,saisie_manuelle,True,ok,True
9993,9994,Fabrica Technologies GmbH,BE,BE1769956921,2025-02-25,import_fournisseur,True,ok,True


In [38]:
df_tva_filtre_1[df_tva_filtre_1["numero_tva"] == "NL785427788B39"]

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite,doublon_tva,doublon_nom
69,70,Vela Materials SpA,NL,NL785427788B39,2024-11-25,reprise_erp,True,ok,False,False
107,108,Vela Materials SpA,NL,NL785427788B39,2024-04-18,crm,True,ok,True,True
759,760,Vela Materials SpA,NL,NL785427788B39,2024-02-23,crm,True,ok,True,True
2019,2020,Vela Materials SpA,NL,NL785427788B39,2025-08-24,crm,True,ok,True,True


In [36]:
df_tva_filtre_1["doublon_nom"] = df_tva_filtre_1["raison_sociale"].duplicated()
df_tva_filtre_1[df_tva_filtre_1["doublon_nom"] == True]

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite,doublon_tva,doublon_nom
101,102,Granit Industries NV,IT,IT53680520532,2024-08-24,saisie_manuelle,True,ok,False,True
103,104,Solaris Technologies SARL,FR,FR53917821806,2025-03-30,crm,True,ok,False,True
107,108,Vela Materials SpA,NL,NL785427788B39,2024-04-18,crm,True,ok,True,True
113,114,Granit Industries NV,IT,IT53680520532,2025-03-06,import_fournisseur,True,ok,True,True
137,138,Lumen Transports BV,PL,PL5102708610,2024-10-03,saisie_manuelle,True,ok,True,True
...,...,...,...,...,...,...,...,...,...,...
9993,9994,Fabrica Technologies GmbH,BE,BE1769956921,2025-02-25,import_fournisseur,True,ok,True,True
9994,9995,Orion Technologies SAS,PT,PT772064113,2024-03-19,portail_client,True,ok,False,True
9995,9996,Cobalt Trading A/S,PT,PT714332224,2025-01-19,import_fournisseur,True,ok,False,True
9996,9997,Delta Trading Oy,FR,FR60801776699,2025-04-18,crm,True,ok,True,True


In [37]:
df_tva_filtre_1[df_tva_filtre_1["raison_sociale"] == "Granit Industries NV"]

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_valide,tva_validite,doublon_tva,doublon_nom
86,87,Granit Industries NV,PT,PT440557399,2025-09-14,reprise_erp,True,ok,False,False
101,102,Granit Industries NV,IT,IT53680520532,2024-08-24,saisie_manuelle,True,ok,False,True
113,114,Granit Industries NV,IT,IT53680520532,2025-03-06,import_fournisseur,True,ok,True,True
5746,5747,Granit Industries NV,LU,LU49021301,2024-03-23,portail_client,True,ok,False,True
6667,6668,Granit Industries NV,NL,852233875B69,2024-04-19,saisie_manuelle,False,pays_non_couvert,False,True
7545,7546,Granit Industries NV,IT,IT53680520532,2024-01-06,saisie_manuelle,True,ok,True,True


- Les doublons de raison sociale peuvent s'expliquer par des présences dans divers pays avec chacun un code tva différent.
- Les doublons de codes TVA s'expliquent par divers types de saisies

Choix :
- Ne garder que la 1ere occurence de chaque numero de tva

In [43]:
df_tva_filtre_2 = df_tva_filtre_1[df_tva_filtre_1["doublon_tva"] == False][["id", "raison_sociale", "pays_declare", "numero_tva", "date_saisie", "source_saisie"]]
df_tva_filtre_2.sample(n=20)

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
2728,2729,Fabrica Services BV,NL,NL735663038B85,2024-03-17,crm
3306,3307,Azur Materials BV,DK,DK75179928,2024-04-28,portail_client
3331,3332,Fabrica Foods BV,IT,IT83849177999,2024-10-10,import_fournisseur
9843,9844,Baltic Technologies Oy,IT,IT81207819012,2025-07-08,reprise_erp
2487,2488,Cobalt Industries BV,FR,FR89874852564,2024-04-23,saisie_manuelle
9504,9505,Metallo Foods SpA,FI,FI76940912,2024-06-15,import_fournisseur
4274,4275,Textil Trading AB,UK,UK69938822,2024-03-18,saisie_manuelle
8776,8777,Cobalt Transports Oy,FR,FR29794733939,2025-08-21,crm
2311,2312,Textil Industries GmbH,NL,NL366225224B14,2024-06-26,saisie_manuelle
5543,5544,Alpine Trading GmbH,UK,UK14148452,2025-04-28,crm


**3. Appel de l'API après nettoyage**

Aucun code tva rentré ne s'avère valide
